# Football Match Prediction — Exploratory Data Analysis

This notebook explores the raw European Soccer Database before modelling:
- Dataset overview and schema
- Class distribution
- Home advantage analysis
- League-level differences
- Feature correlation heatmap
- Seasonal trends

In [ ]:
import sys
sys.path.insert(0, '../src')

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DB_PATH = '../data/raw/database.sqlite'
conn = sqlite3.connect(DB_PATH)
print('Connected to database')

## 1. Database Schema

In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tables:', tables['name'].tolist())

for tbl in tables['name']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {tbl}', conn)
    print(f'  {tbl}: {count["n"][0]:,} rows')

## 2. Load Matches

In [ ]:
matches = pd.read_sql("""
    SELECT m.id, m.season, m.date, m.stage,
           l.name AS league, c.name AS country,
           m.home_team_api_id, m.away_team_api_id,
           m.home_team_goal, m.away_team_goal,
           m.B365H, m.B365D, m.B365A
    FROM Match m
    JOIN League l ON m.league_id = l.id
    JOIN Country c ON m.country_id = c.id
    ORDER BY m.date
""", conn, parse_dates=['date'])

matches = matches.dropna(subset=['home_team_goal', 'away_team_goal'])
matches['home_team_goal'] = matches['home_team_goal'].astype(int)
matches['away_team_goal'] = matches['away_team_goal'].astype(int)

def get_result(row):
    if row.home_team_goal > row.away_team_goal: return 'Home Win'
    elif row.home_team_goal == row.away_team_goal: return 'Draw'
    else: return 'Away Win'

matches['result'] = matches.apply(get_result, axis=1)
print(f'Total matches: {len(matches):,}')
matches.head(3)

## 3. Class Distribution

In [ ]:
result_counts = matches['result'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2196F3', '#FF9800', '#F44336']
bars = ax.bar(result_counts.index, result_counts.values, color=colors, edgecolor='white', width=0.6)
for bar, count in zip(bars, result_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}\n({100*count/len(matches):.1f}%)',
            ha='center', va='bottom', fontsize=11)
ax.set_title('Match Outcome Distribution (All Leagues, All Seasons)')
ax.set_ylabel('Number of Matches')
ax.set_ylim(0, max(result_counts.values) * 1.2)
plt.tight_layout()
plt.show()

## 4. Home Advantage by League

In [ ]:
home_adv = matches.groupby('league')['result'].value_counts(normalize=True).unstack().fillna(0)
home_adv = home_adv.sort_values('Home Win', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
home_adv[['Home Win', 'Draw', 'Away Win']].plot(
    kind='bar', ax=ax, color=['#2196F3', '#FF9800', '#F44336'],
    edgecolor='white', width=0.7
)
ax.set_title('Outcome Distribution by League')
ax.set_ylabel('Proportion')
ax.set_xlabel('')
ax.legend(loc='upper right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 5. Goals per Match Over Seasons

In [ ]:
matches['total_goals'] = matches['home_team_goal'] + matches['away_team_goal']
season_goals = matches.groupby('season')['total_goals'].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(season_goals['season'], season_goals['total_goals'], marker='o',
        color='#1565C0', linewidth=2, markersize=7)
ax.fill_between(season_goals['season'], season_goals['total_goals'],
                alpha=0.15, color='#1565C0')
ax.set_title('Average Goals per Match by Season (All Leagues)')
ax.set_xlabel('Season')
ax.set_ylabel('Avg Goals per Match')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 6. Betting Odds vs Actual Outcomes

Bet365 odds encode the market's implied probabilities.  We can check whether
bookmakers systematically mispriceDraws.

In [ ]:
odds_df = matches[['result', 'B365H', 'B365D', 'B365A']].dropna()

# Convert odds to implied probabilities (and normalise to sum to 1)
odds_df = odds_df.copy()
odds_df['impl_H'] = 1 / odds_df['B365H']
odds_df['impl_D'] = 1 / odds_df['B365D']
odds_df['impl_A'] = 1 / odds_df['B365A']
total = odds_df[['impl_H','impl_D','impl_A']].sum(axis=1)
odds_df['impl_H'] /= total
odds_df['impl_D'] /= total
odds_df['impl_A'] /= total

# Average implied probability vs actual win rate
summary = {
    'Implied P(Home Win)': odds_df['impl_H'].mean(),
    'Actual P(Home Win)':  (odds_df['result']=='Home Win').mean(),
    'Implied P(Draw)':     odds_df['impl_D'].mean(),
    'Actual P(Draw)':      (odds_df['result']=='Draw').mean(),
    'Implied P(Away Win)': odds_df['impl_A'].mean(),
    'Actual P(Away Win)':  (odds_df['result']=='Away Win').mean(),
}
pd.Series(summary).round(4).to_frame('Value')

## 7. Feature Correlation Heatmap

Load the engineered feature matrix and examine correlations with the target.

In [ ]:
import os
feat_path = '../data/processed/features.parquet'

if os.path.exists(feat_path):
    features = pd.read_parquet(feat_path)
    num_cols = features.select_dtypes(include='number').columns.tolist()
    corr = features[num_cols].corr()

    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
                linewidths=0.3, annot=False, ax=ax, vmin=-1, vmax=1)
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
else:
    print('Run src/features.py first to generate the feature matrix.')

In [ ]:
conn.close()
print('Done.')